# Gaussian Splats from a Video (or Frames) — Standalone Init + Train

A distilled, self-contained version of the pipeline in `visergui/splat_trainer.py`
(source line references in each cell). No imports from `visergui`.

**Pipeline**

```
video or frame folder
   └─> Depth-Anything-3  (one joint pass: depth + cameras + per-pixel confidence + sky mask)
         └─> confidence-quantile filter  (keep only the most confident pixels)
               └─> unproject to world points -> voxel downsample -> one gaussian per voxel
                     └─> splats_voxel.ply  (Inria 3DGS format — INITIALIZATION DONE)
                           └─> [optional] gsplat training: masked L1 + void loss
```

Requirements: the `splat` conda env (see `Lyra-2/SPLAT_INSTALL.md`) — for
**initialization** only `torch`, `opencv`, `plyfile`, `matplotlib` and the
`depth_anything_3` package are used; `gsplat` is only needed by the training cells.


In [ ]:
from pathlib import Path

# ---------------- input ----------------
VIDEO_OR_DIR = r"../arch_orbit1.mp4"  # a video file OR a folder of frames (png/jpg)
MAX_FRAMES   = 32                     # uniform-stride frame cap for videos (<=0 = all frames)

# ------------- initialization ----------
CONFIDENCE_QUANTILE = 0.6   # keep pixels above this quantile of DA3 confidence
                            # (0.6 = drop the least-confident 60%, keep the top 40%)
REMOVE_SKY          = True  # drop DA3-detected sky pixels
MAX_POINTS          = 1_000_000
MODEL_ID            = "depth-anything/DA3NESTED-GIANT-LARGE-1.1"
PROCESS_RES         = 504   # DA3 internal inference resolution (lower it if you OOM)
DEVICE              = "cuda"

# ---------------- output ---------------
WORK_DIR = "_work"          # extracted frames, PLYs and diagnostic images land here

# ------------- training ----------------
TRAIN_STEPS  = 0        # 0 = initialization only; training cells become no-ops
MODE         = "3dgs"   # "3dgs" or "2dgs" (2DGS: flat disks + normal/distortion regularizers)
SH_MAX_DEG   = 2        # max spherical-harmonics degree (0 = flat color)
SH_RAMP      = 1000     # steps per SH band unlock (degree ramps 0 -> SH_MAX_DEG)
LPIPS_WEIGHT = 0.05     # perceptual loss weight (0 disables)
USE_DENSIFY  = True     # gsplat clone/split/prune during training


In [ ]:
import math
from dataclasses import dataclass

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from plyfile import PlyData, PlyElement
from tqdm.auto import tqdm

WORK_DIR = Path(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)} | free {free/2**30:.1f} / {total/2**30:.1f} GiB")


@dataclass
class VideoData:
    """Everything DA3 tells us about the clip (all tensors on `device`)."""
    rgb:   torch.Tensor          # (N,H,W,3) float in [0,1]
    depth: torch.Tensor          # (N,H,W)
    K:     torch.Tensor          # (N,3,3) intrinsics, in pixel units of rgb/depth
    w2c:   torch.Tensor          # (N,4,4) OpenCV world->camera
    c2w:   torch.Tensor          # (N,4,4) camera->world
    conf:  torch.Tensor | None   # (N,H,W) per-pixel confidence (arbitrary per-run scale)
    sky:   torch.Tensor | None   # (N,H,W) bool, True where DA3 sees sky
    N: int
    H: int
    W: int


@dataclass
class GaussianInit:
    means:   torch.Tensor        # (G,3) world positions
    quats:   torch.Tensor        # (G,4) wxyz rotations (identity at init)
    log_s:   torch.Tensor        # (G,3) log-scales (isotropic at init)
    logit_o: torch.Tensor        # (G,)  opacity logits
    sh:      torch.Tensor        # (G,1,3) spherical-harmonics DC coefficients
    train_mask: torch.Tensor     # (N,H,W) bool — pixels the training loss may use
    scene_scale: float
    conf_thresh: float | None    # absolute threshold derived from CONFIDENCE_QUANTILE
    voxel: float                 # voxel edge length used for downsampling


## 1 — Load frames

A folder of images is used as-is; a video is sampled at a uniform stride so at most
`MAX_FRAMES` frames survive (distilled from `_extract_frames`, splat_trainer.py:184).
Frames are written at full resolution — DA3 does its own resizing internally.


In [ ]:
def load_frames(video_or_dir, work_dir: Path, max_frames: int = 32) -> list[str]:
    src = Path(video_or_dir)
    if src.is_dir():
        paths = sorted(str(p) for p in src.iterdir()
                       if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
        assert paths, f"no png/jpg images found in {src}"
        print(f"{len(paths)} frames from folder {src}")
        return paths

    assert src.is_file(), f"input not found: {src.resolve()}"
    out_dir = work_dir / "frames"
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(src))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    assert total > 0, (
        f"OpenCV could not decode {src} (unsupported codec?). Extract frames "
        f"yourself (e.g. ffmpeg -i in.mp4 frames/f%04d.png) and pass the folder.")
    stride = max(1, total // max_frames) if max_frames > 0 else 1
    n = max_frames if max_frames > 0 else total
    paths = []
    for i in tqdm(range(n), desc="extracting frames"):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * stride)
        ok, bgr = cap.read()
        if not ok:
            break
        p = str(out_dir / f"f{i:04d}.png")
        cv2.imwrite(p, bgr)
        paths.append(p)
    cap.release()
    print(f"{len(paths)} frames @ stride {stride} from {total} total")
    return paths


## 2 — Depth-Anything-3: depth, cameras and confidence in one pass

One joint DA3 inference over **all** frames gives per-frame depth, intrinsics,
extrinsics, confidence and a sky mask in a single shared coordinate frame — no
COLMAP anywhere (distilled from `_da3_inference_on_paths`, splat_trainer.py:205).
The model is freed immediately after inference to give the GPU memory back.


In [ ]:
def run_da3(frame_paths, model_id: str, process_res: int, device) -> VideoData:
    from depth_anything_3.api import DepthAnything3

    model = DepthAnything3.from_pretrained(model_id).to(device).eval()
    pred = model.inference(
        image=list(frame_paths),
        process_res=process_res,
        process_res_method="upper_bound_resize",
    )

    imgs  = torch.from_numpy(pred.processed_images).to(device)  # (N,H,W,3) uint8
    depth = torch.from_numpy(pred.depth).to(device)             # (N,H,W)
    K     = torch.from_numpy(pred.intrinsics).to(device)        # (N,3,3)
    w2c34 = torch.from_numpy(pred.extrinsics).to(device)        # (N,3,4) OpenCV w2c

    conf_np = getattr(pred, "conf", None)
    if conf_np is None:
        conf_np = getattr(pred, "confidence", None)
    conf = torch.from_numpy(conf_np).to(device) if conf_np is not None else None
    sky_np = getattr(pred, "sky", None)
    sky = torch.from_numpy(sky_np).to(device).bool() if sky_np is not None else None

    N, H, W, _ = imgs.shape
    w2c = torch.eye(4, device=device).expand(N, 4, 4).clone()
    w2c[:, :3, :4] = w2c34
    c2w = torch.linalg.inv(w2c)
    rgb = imgs.float() / 255.0

    del model, imgs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"DA3 over {N} frames: H={H} W={W}; "
          f"depth range [{depth.min():.3f}, {depth.max():.3f}]")
    return VideoData(rgb=rgb, depth=depth, K=K, w2c=w2c, c2w=c2w,
                     conf=conf, sky=sky, N=N, H=H, W=W)


In [ ]:
frame_paths = load_frames(VIDEO_OR_DIR, WORK_DIR, MAX_FRAMES)
data = run_da3(frame_paths, MODEL_ID, PROCESS_RES, device)
print(f"rgb {tuple(data.rgb.shape)}  depth {tuple(data.depth.shape)}  K {tuple(data.K.shape)}")
print(f"confidence: {'yes' if data.conf is not None else 'NO'}   "
      f"sky mask: {'yes' if data.sky is not None else 'NO'}")


## 3 — The confidence threshold

`CONFIDENCE_QUANTILE` is a **quantile, not an absolute value**: DA3 confidence is on
a different absolute scale every run, so the threshold is derived from this clip's
own confidences (`_conf_quantile_thresh`, splat_trainer.py:554). The same threshold
gates both which pixels become gaussians **and** which pixels the training loss is
allowed to supervise (`_per_frame_train_mask`, splat_trainer.py:534).


In [ ]:
def conf_quantile_thresh(conf: torch.Tensor, q: float) -> float:
    """Absolute confidence value at quantile q of this clip's confidences."""
    flat = conf.flatten()
    if flat.numel() > 16_000_000:  # bound quantile() memory on long clips
        flat = flat[torch.randint(0, flat.numel(), (16_000_000,), device=flat.device)]
    return float(flat.quantile(q))


def make_train_mask(depth, sky, conf, conf_thresh, remove_sky) -> torch.Tensor:
    """Validity gate: depth > 0, not sky, confidence above the threshold."""
    mask = depth > 0
    if sky is not None and remove_sky:
        mask &= ~sky
    if conf is not None and conf_thresh is not None:
        mask &= conf > conf_thresh
    return mask


In [ ]:
conf_thresh = conf_quantile_thresh(data.conf, CONFIDENCE_QUANTILE) if data.conf is not None else None

show = np.linspace(0, data.N - 1, min(4, data.N)).round().astype(int)
fig, axes = plt.subplots(len(show), 4, figsize=(16, 3.2 * len(show)), squeeze=False)
for r, idx in enumerate(show):
    idx = int(idx)
    axes[r, 0].imshow(data.rgb[idx].cpu().numpy())
    axes[r, 0].set_title(f"frame {idx}")
    axes[r, 1].imshow(data.depth[idx].cpu().numpy(), cmap="turbo")
    axes[r, 1].set_title("DA3 depth")
    if data.conf is not None:
        axes[r, 2].imshow(data.conf[idx].cpu().numpy(), cmap="viridis")
    axes[r, 2].set_title("DA3 confidence")
    mask = make_train_mask(data.depth[idx],
                           None if data.sky is None else data.sky[idx],
                           None if data.conf is None else data.conf[idx],
                           conf_thresh, REMOVE_SKY)
    axes[r, 3].imshow(mask.cpu().numpy(), cmap="gray")
    axes[r, 3].set_title(f"kept pixels @ q={CONFIDENCE_QUANTILE}")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(WORK_DIR / "init_diagnostics.png", dpi=110)
plt.show()


## 4 — Initialize the gaussians

Every valid pixel is unprojected to a world point (`x = c2w @ (K⁻¹ [u,v,1]ᵀ · z)`),
then the cloud is voxel-downsampled so overlapping frames don't create redundant
co-located splats — **one gaussian per occupied voxel**, with position/color averaged
over the pixels that landed in it (distilled from `_build_initial_gaussians`,
splat_trainer.py:568). Each gaussian starts isotropic (scale ≈ one source texel at
its depth, at least half a voxel so neighbours overlap), 90% opaque, unrotated.


In [ ]:
def scatter_mean(vals, inv, g):
    """Mean of `vals` per group in `inv` (splat_trainer.py:468)."""
    counts = torch.zeros(g, device=vals.device).scatter_add_(
        0, inv, torch.ones(vals.shape[0], device=vals.device)).clamp_min(1)
    if vals.ndim == 1:
        return torch.zeros(g, device=vals.device).scatter_add_(0, inv, vals) / counts
    D = vals.shape[1]
    s = torch.zeros(g, D, device=vals.device).scatter_add_(
        0, inv[:, None].expand(-1, D), vals)
    return s / counts[:, None]


def build_gaussians(data: VideoData, q: float, remove_sky: bool,
                    max_points: int, voxel_frac: float = 0.005) -> GaussianInit:
    device = data.rgb.device

    # 1. validity mask: depth > 0, not sky, confidence above the q-quantile
    conf_thresh = conf_quantile_thresh(data.conf, q) if data.conf is not None else None
    valid = make_train_mask(data.depth, data.sky, data.conf, conf_thresh, remove_sky)

    # 2. unproject every pixel of every frame to world space (:576-584)
    ii, jj = torch.meshgrid(torch.arange(data.H, device=device),
                            torch.arange(data.W, device=device), indexing="ij")
    uv1 = torch.stack([jj, ii, torch.ones_like(ii)], -1).float()            # (H,W,3)
    cam_pts = torch.einsum("nij,hwj->nhwi",
                           torch.linalg.inv(data.K), uv1) * data.depth[..., None]
    R, t = data.c2w[:, :3, :3], data.c2w[:, :3, 3]
    world = torch.einsum("nij,nhwj->nhwi", R, cam_pts) + t[:, None, None, :]

    pts  = world[valid]                                                     # (T,3)
    cols = data.rgb[valid]                                                  # (T,3)
    z    = data.depth[valid]
    fx   = data.K[:, 0, 0][:, None, None].expand(data.N, data.H, data.W)[valid]
    T = pts.shape[0]
    assert T > 0, "the confidence threshold removed every pixel — lower CONFIDENCE_QUANTILE"

    scene_scale = float(pts.std(dim=0).mean())
    voxel = max(scene_scale * voxel_frac, 1e-4)

    # 3. voxel downsample: one gaussian per occupied voxel (:639-650)
    keys = torch.floor(pts / voxel).long()
    uniq, inv = torch.unique(keys, dim=0, return_inverse=True)
    G = uniq.shape[0]
    means  = scatter_mean(pts,  inv, G)
    color  = scatter_mean(cols, inv, G)
    z_vox  = scatter_mean(z,    inv, G)
    fx_vox = scatter_mean(fx,   inv, G)

    # 4. per-gaussian attributes (:652-661) — scale is the larger of one source
    #    texel at that depth (inflated for sparsity) and half the voxel edge
    inflate = max(1.0, (T / G) ** 0.5)
    tex = (z_vox / fx_vox * inflate).clamp_min(voxel * 0.5)
    C0 = 0.28209479177387814                      # SH DC basis constant
    quats = torch.zeros((G, 4), device=device)
    quats[:, 0] = 1.0                             # identity rotation (wxyz)
    init = GaussianInit(
        means=means,
        quats=quats,
        log_s=torch.log(tex[:, None].expand(G, 3).contiguous()),
        logit_o=torch.full((G,), 2.1972, device=device),   # sigmoid^-1(0.9)
        sh=((color - 0.5) / C0)[:, None, :],
        train_mask=valid,
        scene_scale=scene_scale, conf_thresh=conf_thresh, voxel=voxel,
    )

    # 5. honor the point budget (:668-673)
    if G > max_points:
        sel = torch.randperm(G, device=device)[:max_points]
        init.means, init.quats, init.log_s = init.means[sel], init.quats[sel], init.log_s[sel]
        init.logit_o, init.sh = init.logit_o[sel], init.sh[sel]
        G = max_points
        print(f"voxel init exceeded MAX_POINTS — subsampled to {G}")

    print(f"{G} gaussians from {T} valid pixels "
          f"({100 * valid.float().mean():.1f}% of pixels kept)")
    print(f"scene_scale={scene_scale:.3f}  voxel={voxel:.4f}  conf_thresh={conf_thresh}")
    return init


In [ ]:
def save_inria_ply(path, means, sh_all, log_s, logit_o, quats):
    """Inria-format 3DGS PLY (splat_trainer.py:920) — loads in standard 3DGS
    viewers. sh_all: (G, K, 3); scales stay in log space, opacity stays a logit."""
    n = means.shape[0]
    K_rest = sh_all.shape[1] - 1
    fields = ([("x", "f4"), ("y", "f4"), ("z", "f4"),
               ("nx", "f4"), ("ny", "f4"), ("nz", "f4"),
               ("f_dc_0", "f4"), ("f_dc_1", "f4"), ("f_dc_2", "f4")]
              + [(f"f_rest_{i}", "f4") for i in range(3 * K_rest)]
              + [("opacity", "f4"),
                 ("scale_0", "f4"), ("scale_1", "f4"), ("scale_2", "f4"),
                 ("rot_0", "f4"), ("rot_1", "f4"), ("rot_2", "f4"), ("rot_3", "f4")])
    arr = np.zeros(n, dtype=fields)
    m = means.detach().cpu().numpy()
    sh_np = sh_all.detach().cpu().numpy()
    s = log_s.detach().cpu().numpy()
    q = quats.detach().cpu().numpy()
    arr["x"], arr["y"], arr["z"] = m[:, 0], m[:, 1], m[:, 2]
    arr["f_dc_0"], arr["f_dc_1"], arr["f_dc_2"] = sh_np[:, 0, 0], sh_np[:, 0, 1], sh_np[:, 0, 2]
    if K_rest > 0:
        rest = sh_np[:, 1:, :].transpose(0, 2, 1).reshape(n, 3 * K_rest)
        for i in range(3 * K_rest):
            arr[f"f_rest_{i}"] = rest[:, i]
    arr["opacity"] = logit_o.detach().cpu().numpy()
    arr["scale_0"], arr["scale_1"], arr["scale_2"] = s[:, 0], s[:, 1], s[:, 2]
    arr["rot_0"], arr["rot_1"], arr["rot_2"], arr["rot_3"] = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    PlyData([PlyElement.describe(arr, "vertex")], text=False).write(str(path))


init = build_gaussians(data, CONFIDENCE_QUANTILE, REMOVE_SKY, MAX_POINTS)

# sanity checks (the smoke harness re-verifies these against the PLY on disk)
assert init.means.shape[0] > 0
for name in ("means", "quats", "log_s", "logit_o", "sh"):
    assert torch.isfinite(getattr(init, name)).all(), f"non-finite values in {name}"

ply_path = WORK_DIR / "splats_voxel.ply"
save_inria_ply(ply_path, init.means, init.sh, init.log_s, init.logit_o, init.quats)
print(f"saved {ply_path}  ({init.means.shape[0]} gaussians)")


In [ ]:
n_show = min(20_000, init.means.shape[0])
sel = torch.randperm(init.means.shape[0])[:n_show]
pts = init.means[sel].cpu().numpy()
col = (init.sh[sel, 0] * 0.28209479177387814 + 0.5).clamp(0, 1).cpu().numpy()

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection="3d")
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=col, s=0.5)
lo, hi = np.percentile(pts, [1, 99], axis=0)      # robust equal-aspect box
ctr, ext = (lo + hi) / 2, (hi - lo).max() / 2
ax.set_xlim(ctr[0] - ext, ctr[0] + ext)
ax.set_ylim(ctr[1] - ext, ctr[1] + ext)
ax.set_zlim(ctr[2] - ext, ctr[2] + ext)
ax.view_init(elev=-60, azim=-90)                  # roughly camera-forward (OpenCV frame)
ax.set_title(f"initialized gaussians ({init.means.shape[0]} total, {n_show} shown)")
fig.savefig(WORK_DIR / "init_scatter.png", dpi=110)
plt.show()


---
## ✅ Initialization complete

`splats_voxel.ply` above is a valid 3DGS scene you can already open in any Inria-PLY
viewer (e.g. `visergui/native_viewer.py --ply _work/splats_voxel.ply`).

Everything below is **training** and additionally needs `gsplat`
(CUDA-only; on Windows its first run JIT-compiles kernels, which requires MSVC and
a CUDA toolkit). Set `TRAIN_STEPS > 0` to enable it.


In [ ]:
def make_train_state(init: GaussianInit, sh_max_deg: int = 0,
                     use_densify: bool = False, total_steps: int = 7000,
                     mode: str = "3dgs"):
    """ParameterDict + ONE Adam per parameter (splat_trainer.py:702-775).
    Per-key optimizers are required by gsplat's DefaultStrategy so it can rebuild
    Adam state when the splat count changes during clone/split/prune.
    shN holds the higher SH bands: (M, (deg+1)^2 - 1, 3), zero-width at deg 0."""
    torch.set_grad_enabled(True)
    M = init.means.shape[0]
    K_rest = (sh_max_deg + 1) ** 2 - 1
    params = torch.nn.ParameterDict({
        "means":     torch.nn.Parameter(init.means.clone()),
        "scales":    torch.nn.Parameter(init.log_s.clone()),
        "quats":     torch.nn.Parameter(init.quats.clone()),
        "opacities": torch.nn.Parameter(init.logit_o.clone()),
        "sh0":       torch.nn.Parameter(init.sh.clone()),
        "shN":       torch.nn.Parameter(
            torch.zeros((M, K_rest, 3), device=init.means.device)),
    })
    lr_table = {"means": 1.6e-4 * init.scene_scale if use_densify else 1.6e-4,
                "scales": 5e-3, "quats": 1e-3, "opacities": 5e-2,
                "sh0": 2.5e-3, "shN": 2.5e-3 / 20.0}
    opts = {k: torch.optim.Adam([{"params": [params[k]], "lr": lr}])
            for k, lr in lr_table.items()}

    strategy = strategy_state = None
    if use_densify:
        from gsplat import DefaultStrategy
        if mode == "2dgs":
            # 2DGS recipe (:750-760): opacity reset disabled (reset_every > steps),
            # gradient key switched to the 2DGS accumulator
            strategy = DefaultStrategy(
                refine_start_iter=500,
                refine_stop_iter=max(500, total_steps - 500),
                reset_every=total_steps + 1, refine_every=100,
                key_for_gradient="gradient_2dgs", verbose=False)
        else:
            # aggressive 3DGS schedule (:762-770)
            strategy = DefaultStrategy(
                refine_start_iter=200,
                refine_stop_iter=max(500, total_steps - 200),
                reset_every=1500, refine_every=50, verbose=False)
        strategy.check_sanity(params, opts)
        strategy_state = strategy.initialize_state(scene_scale=init.scene_scale)
    return params, opts, strategy, strategy_state


def train(params, opts, data: VideoData, init: GaussianInit, steps: int,
          mode: str = "3dgs", strategy=None, strategy_state=None,
          sh_max_deg: int = 0, sh_ramp: int = 1000,
          l1_weight: float = 1.0, void_weight: float = 0.5,
          lpips_weight: float = 0.0, log_scale_max: float | None = None,
          distortion_weight: float = 1.0, dist_warmup: int = 700,
          normal_weight: float = 0.05, normal_warmup: int = 1600,
          da3_normals: torch.Tensor | None = None,
          depth_sup_weight: float = 0.5, da3_normal_weight: float = 0.05):
    """Training loop distilled from splat_trainer.step() (:2746-2888).
    One random camera per step. Losses: masked L1 + void alpha^2 (outside the
    supervision mask) + optional LPIPS; 2DGS additionally gets distortion and
    normal-consistency regularizers after their warmups, plus DA3 depth and
    normal supervision (active from step 0 — DA3 is the init source, these keep
    the disks anchored to those surfaces). SH degree ramps 0 -> sh_max_deg,
    one band per sh_ramp steps (:1255)."""
    from gsplat import rasterization, rasterization_2dgs

    lpips_net = None
    if lpips_weight > 0:
        import lpips as lpips_mod
        lpips_net = lpips_mod.LPIPS(net="alex").to(data.rgb.device)

    losses = []
    pbar = tqdm(range(steps), desc=f"training ({mode})")
    for step in pbar:
        idx = int(torch.randint(0, data.N, (1,)).item())
        cur_sh = min(sh_max_deg, step // sh_ramp) if sh_ramp > 0 else sh_max_deg
        sh_all = torch.cat([params["sh0"], params["shN"]], dim=1)  # no-op at deg 0

        normals = surf_normals = distort = depth_pred = None
        if mode == "2dgs":
            colors, alphas, normals, surf_normals, distort, _, info = rasterization_2dgs(
                params["means"], params["quats"], torch.exp(params["scales"]),
                torch.sigmoid(params["opacities"]), sh_all,
                data.w2c[idx:idx + 1], data.K[idx:idx + 1], data.W, data.H,
                sh_degree=cur_sh, packed=False, render_mode="RGB+ED", distloss=True)
            pred = colors[0, ..., :3]
            depth_pred = colors[0, ..., 3]                   # rendered depth ("ED")
        else:
            out, alphas, info = rasterization(
                params["means"], params["quats"], torch.exp(params["scales"]),
                torch.sigmoid(params["opacities"]), sh_all,
                data.w2c[idx:idx + 1], data.K[idx:idx + 1], data.W, data.H,
                sh_degree=cur_sh, packed=False)
            pred = out[0]                                    # (H,W,3)

        if strategy is not None:                             # densify bookkeeping
            strategy.step_pre_backward(params, opts, strategy_state, step, info)

        mask = init.train_mask[idx].unsqueeze(-1)
        n_kept = int(mask.sum())
        if n_kept == 0:
            continue
        loss = l1_weight * ((pred - data.rgb[idx]).abs() * mask).sum() / (n_kept * 3)

        # void loss: push rendered alpha to zero on masked-out (sky / low-conf)
        # pixels so boundary splats can't grow into unsupervised space for free
        void_px = (~init.train_mask[idx]).float()
        n_void = int(void_px.sum())
        if void_weight > 0 and n_void > 0:
            loss = loss + void_weight * (alphas[0, ..., 0].pow(2) * void_px).sum() / n_void

        # LPIPS on the full frame (masking a perceptual loss makes no sense, :2820)
        if lpips_net is not None:
            lp = lpips_net(pred.permute(2, 0, 1)[None] * 2 - 1,
                           data.rgb[idx].permute(2, 0, 1)[None] * 2 - 1).mean()
            loss = loss + lpips_weight * lp

        # 2DGS regularizers, warmup-gated so disks learn RGB before flattening (:2835)
        if mode == "2dgs":
            if distort is not None and step >= dist_warmup:
                loss = loss + distortion_weight * distort.mean()
            if normals is not None and surf_normals is not None and step >= normal_warmup:
                loss = loss + normal_weight * (1 - (normals * surf_normals).sum(-1)).mean()
            # DA3 depth + normal supervision (:2847-2863): L1 between rendered
            # and DA3 depth, and 1-cos between rendered normals and normals
            # derived from DA3 depth, wherever DA3 actually saw a surface
            zmask = data.depth[idx] > 0.01
            if depth_sup_weight > 0 and depth_pred is not None and zmask.any():
                loss = loss + depth_sup_weight * (depth_pred - data.depth[idx]).abs()[zmask].mean()
            if da3_normals is not None and normals is not None and zmask.any():
                cos = (normals[0] * da3_normals[idx]).sum(-1)
                loss = loss + da3_normal_weight * (1 - cos[zmask]).mean()

        loss.backward()
        for o in opts.values():
            o.step()
            o.zero_grad(set_to_none=True)
        if strategy is not None:                             # clone/split/prune (:2879)
            strategy.step_post_backward(params, opts, strategy_state, step, info,
                                        packed=False)
        if log_scale_max is not None and mode == "3dgs":     # cap splat growth (:2886)
            with torch.no_grad():
                params["scales"].clamp_(max=log_scale_max)

        losses.append(float(loss))
        if step % 20 == 0:
            pbar.set_postfix(loss=f"{losses[-1]:.4f}",
                             splats=params["means"].shape[0], sh=cur_sh)
    return losses


In [ ]:
losses = None
if TRAIN_STEPS > 0:
    try:
        import gsplat  # noqa: F401 — first use JIT-compiles CUDA kernels (~1 min)
    except ImportError as e:
        print(f"gsplat unavailable ({e}); skipping training.\n"
              "Install with: pip install gsplat==1.5.3 "
              "(needs MSVC + CUDA 12.8 toolkit on Windows).")
    else:
        params, opts, strategy, strategy_state = make_train_state(
            init, sh_max_deg=SH_MAX_DEG, use_densify=USE_DENSIFY,
            total_steps=TRAIN_STEPS, mode=MODE)
        # 2DGS: precompute per-pixel world-space normals from DA3 depth, same
        # convention as gsplat's surf_normals (splat_trainer.py:1227-1235)
        da3_normals = None
        if MODE == "2dgs":
            from gsplat.utils import depth_to_normal
            with torch.no_grad():
                da3_normals = depth_to_normal(data.depth[..., None], data.c2w, data.K)
        losses = train(params, opts, data, init, TRAIN_STEPS, mode=MODE,
                       strategy=strategy, strategy_state=strategy_state,
                       sh_max_deg=SH_MAX_DEG, sh_ramp=SH_RAMP,
                       lpips_weight=LPIPS_WEIGHT, da3_normals=da3_normals,
                       log_scale_max=math.log(2.0 * init.voxel))
        print(f"splats after training: {params['means'].shape[0]} "
              f"(init had {init.means.shape[0]})")
else:
    print("TRAIN_STEPS = 0 — init-only run, training skipped.")


In [ ]:
if losses:
    trained_path = WORK_DIR / "splats_trained.ply"
    sh_full = torch.cat([params["sh0"], params["shN"]], dim=1)   # DC + higher bands
    save_inria_ply(trained_path, params["means"], sh_full,
                   params["scales"], params["opacities"], params["quats"])
    print(f"saved {trained_path}  ({params['means'].shape[0]} gaussians, "
          f"SH degree {SH_MAX_DEG})")

    plt.figure(figsize=(7, 3))
    plt.plot(losses)
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title("training loss")
    plt.tight_layout()
    plt.savefig(WORK_DIR / "train_loss.png", dpi=110)
    plt.show()


## Adapting this notebook

Knobs worth playing with:

- **`CONFIDENCE_QUANTILE`** — higher = fewer, more reliable points (and less supervised
  image area); lower = denser but noisier init.
- **`voxel_frac`** (arg of `build_gaussians`, default 0.005·scene_scale) — smaller voxels
  = more, finer gaussians.
- **`MAX_FRAMES` / `PROCESS_RES` / `MODEL_ID`** — the VRAM levers, in order of impact.

Training features and their key knobs:

- **Densification** (`USE_DENSIFY`) — gsplat `DefaultStrategy` clone/split/prune;
  the splat count changes during training (watch the `splats=` progress readout).
- **SH degrees** (`SH_MAX_DEG`, `SH_RAMP`) — view-dependent color; bands unlock
  gradually so the DC color stabilizes first.
- **LPIPS** (`LPIPS_WEIGHT`) — perceptual loss over the full frame; helps texture
  sharpness at a per-step speed cost.
- **2DGS** (`MODE="2dgs"`) — flat disks instead of ellipsoids, with distortion and
  normal-consistency regularizers plus DA3 depth/normal supervision (rendered
  depth is pulled toward DA3 depth, disk normals toward normals derived from it);
  better surfaces, e.g. for later meshing.

Still deliberately omitted from the full trainer (`visergui/splat_trainer.py`):

| Feature | Where to find it |
|---|---|
| Appending new clips to an existing scene | `append_video`, :1532 |
| Checkpoint save/load with metadata | `save_checkpoint` / `load_checkpoint`, :1329 |
